# Dataset Quality Investigation

**Purpose.** This notebook inspects five real-world datasets and records basic data-quality
observations for each one: **missing data**, **incomplete data** (including documented
nonresponse codes) and **duplicate data**. It is descriptive only. No values are cleaned, imputed,
recoded or removed, and no modelling or statistical testing is performed.

| # | Dataset | Publisher | Official source |
|---|---------|-----------|-----------------|
| 1 | NHANES Body Measures (`BMX_L`, 2021–2023) | CDC / NCHS | <https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm> |
| 2 | NHANES Dietary Interview – Total Nutrient Intakes, First Day (`DR1TOT_L`, 2021–2023) | CDC / NCHS | <https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1TOT_L.htm> |
| 3 | BRFSS 2024 | CDC | <https://www.cdc.gov/brfss/annual_data/annual_2024.html> |
| 4 | Fuel Economy Vehicle Data | U.S. EPA / DOE | <https://www.fueleconomy.gov/feg/ws/index.shtml> |
| 5 | Amazon Purchase Histories (Open e-commerce 1.0) | Harvard Dataverse | <https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY> |

**How the notebook is organised.** Every dataset goes through the same six steps:

1. Load the dataset from its official source
2. Display a representative excerpt
3. Investigate missing data
4. Investigate incomplete data
5. Investigate duplicate data
6. Summarise the findings

A comparison table for all five datasets is built at the end.

**Practical notes**

- All files are downloaded directly from the publishers listed above and cached in a local `data/`
  folder, so re-running the notebook does not download them again. About 1.5 GB of disk space is
  needed (most of it is the unzipped BRFSS file) and roughly 4 GB of free memory.
- Every analysis uses the **complete** dataset. Only the *displayed excerpts* are subsets
  (a few rows and, for wide datasets, a few columns).
- Required packages: `pandas`, `requests`, `pyreadstat`. On Python 3.9 install
  `pyreadstat<1.3` (newer releases need Python 3.10+).
- The value labels and codes quoted in the text come from the official codebooks linked in each
  section. All counts are calculated by the code cells from the downloaded data.

In [1]:
# Uncomment if pyreadstat is not installed yet (use "pyreadstat<1.3" on Python 3.9)
# %pip install "pyreadstat<1.3"

In [2]:
import gc
import hashlib
import zipfile
from datetime import date
from pathlib import Path

import pandas as pd
import pyreadstat
import requests

DATA_DIR = Path("data")      # downloaded files are cached here
DATA_DIR.mkdir(exist_ok=True)

SEED = 42                    # fixed seed so the displayed excerpts are reproducible
EXCERPT_ROWS = 8

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)   # only shortens long text on screen, not the data
pd.set_option("display.width", 200)

print("Notebook run date:", date.today())
print("pandas", pd.__version__, "| pyreadstat", pyreadstat.__version__)

Notebook run date: 2026-10-10
pandas 1.4.4 | pyreadstat 1.2.7


## Reusable helper functions

The same small functions are used for all five datasets so that the results are reported in a
consistent format.

In [3]:
def download_file(url, path, manual_hint):
    # Download `url` to `path` once. Stops with a clear message if the download fails.
    path = Path(path)
    if path.exists():
        print(f"Using cached file: {path} ({path.stat().st_size / 1e6:,.1f} MB)")
        return path
    part = path.with_name(path.name + ".part")   # write to a temporary name first
    try:
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with open(part, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
        part.rename(path)
    except Exception as err:
        raise RuntimeError(
            f"Could not download {url}\nReason: {err}\n"
            f"To continue, obtain the original file manually: {manual_hint}"
        ) from err
    print(f"Downloaded: {path} ({path.stat().st_size / 1e6:,.1f} MB)")
    return path

In [4]:
def show_dimensions(df, name, source):
    # Print the dataset name, official source and size.
    print(f"Dataset : {name}")
    print(f"Source  : {source}")
    print(f"Rows    : {len(df):,}")
    print(f"Columns : {df.shape[1]:,}")


def show_columns(df, labels=None, max_table_rows=30):
    # Show column names and data types (a compact list is used for very wide datasets).
    print("Data types:", df.dtypes.astype(str).value_counts().to_dict())
    if df.shape[1] <= max_table_rows:
        table = pd.DataFrame({"dtype": df.dtypes.astype(str)})
        if labels:
            table["label"] = [labels.get(col, "") for col in df.columns]
        display(table)
    else:
        print(f"\nAll {df.shape[1]} column names:")
        print(", ".join(f"{col} ({dtype})" for col, dtype in df.dtypes.astype(str).items()))


def show_excerpt(df, columns, n=EXCERPT_ROWS):
    # Return n randomly chosen original records (fixed seed), in their original order.
    return df[columns].sample(n=n, random_state=SEED).sort_index()

In [5]:
def missing_table(df):
    # Missing-value count and percentage for every column, largest first.
    counts = df.isna().sum()
    table = pd.DataFrame({
        "missing_count": counts,
        "missing_pct": (counts / len(df) * 100).round(2),
    })
    return table.sort_values("missing_count", ascending=False)


def missing_overview(df):
    # Print and return the overall missing-value figures for a dataset.
    counts = df.isna().sum()
    overview = {
        "total_cells": df.size,
        "total_missing": int(counts.sum()),
        "missing_pct": round(counts.sum() / df.size * 100, 2),
        "columns_with_missing": int((counts > 0).sum()),
        "columns_all_missing": int((counts == len(df)).sum()),
    }
    print(f"Total cells                    : {overview['total_cells']:,}")
    print(f"Total missing values           : {overview['total_missing']:,} ({overview['missing_pct']}% of all cells)")
    print(f"Columns with missing values    : {overview['columns_with_missing']} of {df.shape[1]}")
    print(f"Columns that are entirely empty: {overview['columns_all_missing']}")
    return overview

In [6]:
def special_code_table(df, codes):
    # Count documented special codes. `codes` maps column -> {code: meaning from the codebook}.
    rows = []
    for column, code_meanings in codes.items():
        for code, meaning in code_meanings.items():
            count = int((df[column] == code).sum())
            rows.append({
                "column": column,
                "code": code,
                "documented meaning": meaning,
                "count": count,
                "pct_of_rows": round(count / len(df) * 100, 2),
            })
    return pd.DataFrame(rows)


def duplicate_overview(df, id_columns=None):
    # Print and return exact-duplicate-row and duplicate-identifier counts.
    exact = int(df.duplicated().sum())
    overview = {"exact_duplicates": exact,
                "exact_duplicates_pct": round(exact / len(df) * 100, 3)}
    print(f"Exact duplicate rows: {exact:,} ({overview['exact_duplicates_pct']}% of rows)")
    if id_columns:
        id_dups = int(df.duplicated(subset=id_columns).sum())
        overview["id_duplicates"] = id_dups
        print(f"Rows repeating an earlier identifier {id_columns}: {id_dups:,}")
    return overview

In [7]:
PLACEHOLDERS = ["", "na", "n/a", "nan", "null", "none", "unknown", "-", "?"]


def placeholder_table(df):
    # Count text values that look like a stand-in for a missing value (case and spaces ignored).
    rows = []
    for column in df.select_dtypes(include="object").columns:
        values = df[column].dropna()
        matches = values[values.str.strip().str.lower().isin(PLACEHOLDERS)]
        for value, count in matches.value_counts().items():
            rows.append({"column": column, "text value": value, "count": int(count)})
    return pd.DataFrame(rows, columns=["column", "text value", "count"])

In [8]:
RESULTS = []   # one entry per dataset, used for the final comparison table


def record_results(name, df, missing, duplicates, observations):
    # Store the calculated figures and print the per-dataset summary.
    RESULTS.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Missing Values (%)": missing["missing_pct"],
        "Exact Duplicate Rows": duplicates["exact_duplicates"],
        "Main Data-Quality Observations": " ".join(observations),
    })
    print(f"Summary - {name}")
    for line in observations:
        print(" -", line)

---
# Dataset 1 — NHANES Body Measures (`BMX_L`, 2021–2023)

- **Publisher:** CDC / National Center for Health Statistics
- **Documentation and codebook:** <https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm>
- **Data file:** `BMX_L.xpt` (SAS transport file) from the same CDC page
- **Unit of observation:** one row per examined survey participant (`SEQN`)

## Step 1: Load the dataset

NHANES files are SAS transport (`.xpt`) files. They are read with `pyreadstat`, which also returns
the official variable labels. `pandas.read_sas` is not used because, for these files, it returns
true zeros as `5.4e-79`, which would not preserve the original values.

In [9]:
BMX_NAME = "NHANES Body Measures (BMX_L, 2021-2023)"
BMX_SOURCE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm"
BMX_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt"

bmx_path = download_file(
    BMX_URL, DATA_DIR / "BMX_L.xpt",
    manual_hint=f"open {BMX_SOURCE}, download the XPT data file and save it as data/BMX_L.xpt",
)
bmx, bmx_meta = pyreadstat.read_xport(str(bmx_path))
bmx_labels = bmx_meta.column_names_to_labels

show_dimensions(bmx, BMX_NAME, BMX_SOURCE)

Using cached file: data/BMX_L.xpt (1.6 MB)
Dataset : NHANES Body Measures (BMX_L, 2021-2023)
Source  : https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.htm
Rows    : 8,860
Columns : 22


In [10]:
show_columns(bmx, labels=bmx_labels)

Data types: {'float64': 22}


,dtype,label
SEQN,float64,Respondent sequence number
BMDSTATS,float64,Body Measures Component Status Code
BMXWT,float64,Weight (kg)
BMIWT,float64,Weight Comment
BMXRECUM,float64,Recumbent Length (cm)
BMIRECUM,float64,Recumbent Length Comment
BMXHEAD,float64,Head Circumference (cm)
BMIHEAD,float64,Head Circumference Comment
BMXHT,float64,Standing Height (cm)
BMIHT,float64,Standing Height Comment


All columns are stored as floating-point numbers because the SAS transport format has a single
numeric type. Identifiers and codes such as `SEQN` or `BMDSTATS` therefore display with `.0`.

## Step 2: Representative excerpt

The excerpt shows original records chosen at random with a fixed seed. Only a subset of the 22
columns is shown for readability (the measurements most often used, plus one comment column).

In [11]:
bmx_excerpt_cols = ["SEQN", "BMDSTATS", "BMXWT", "BMIWT", "BMXHT", "BMXBMI",
                    "BMDBMIC", "BMXARMC", "BMXWAIST", "BMXHIP"]
show_excerpt(bmx, bmx_excerpt_cols)

,SEQN,BMDSTATS,BMXWT,BMIWT,BMXHT,BMXBMI,BMDBMIC,BMXARMC,BMXWAIST,BMXHIP
222,130673.0,1.0,41.2,NaN,146.7,19.1,2.0,23.6,67.2,80.7
3859,135587.0,1.0,61.1,NaN,155.7,25.2,NaN,29.0,97.4,99.2
4011,135797.0,1.0,73.8,NaN,166.6,26.6,NaN,28.1,97.9,106.6
4890,136987.0,1.0,60.0,NaN,152.3,25.9,NaN,30.2,89.0,97.0
5952,138407.0,1.0,37.8,NaN,131.8,21.8,4.0,24.3,71.1,NaN
6280,138843.0,1.0,54.4,NaN,167.2,19.5,2.0,25.4,69.5,94.4
6347,138936.0,1.0,102.1,NaN,172.6,34.3,NaN,34.5,115.4,115.4
7768,140845.0,1.0,67.5,NaN,161.5,25.9,NaN,27.5,89.5,105.2


## Step 3: Missing data

In [12]:
bmx_missing = missing_overview(bmx)

Total cells                    : 194,920
Total missing values           : 107,023 (54.91% of all cells)
Columns with missing values    : 20 of 22
Columns that are entirely empty: 1


The codebook gives a **target age range** for each measurement (for example, head circumference is
only measured from birth to 6 months). Participants outside the range have no value by design. The
table below places the documented target next to the missing-value figures.

In [13]:
# Target populations transcribed from the BMX_L codebook
BMX_TARGET = {
    "SEQN": "0-150 years", "BMDSTATS": "0-150 years",
    "BMXWT": "0-150 years", "BMIWT": "0-150 years",
    "BMXRECUM": "0-47 months", "BMIRECUM": "0-47 months",
    "BMXHEAD": "0-6 months", "BMIHEAD": "0-6 months",
    "BMXHT": "2-150 years", "BMIHT": "2-150 years",
    "BMXBMI": "2-150 years", "BMDBMIC": "2-19 years",
    "BMXLEG": "8-150 years", "BMILEG": "8-150 years",
    "BMXARML": "2 months-150 years", "BMIARML": "2 months-150 years",
    "BMXARMC": "2 months-150 years", "BMIARMC": "2 months-150 years",
    "BMXWAIST": "2-150 years", "BMIWAIST": "2-150 years",
    "BMXHIP": "12-150 years", "BMIHIP": "12-150 years",
}

bmx_missing_table = missing_table(bmx)
bmx_missing_table["label"] = bmx_missing_table.index.map(bmx_labels)
bmx_missing_table["documented target"] = bmx_missing_table.index.map(BMX_TARGET)
bmx_missing_table

,missing_count,missing_pct,label,documented target
BMIHEAD,8860,100.00,Head Circumference Comment,0-6 months
BMIRECUM,8842,99.80,Recumbent Length Comment,0-47 months
BMXHEAD,8790,99.21,Head Circumference (cm),0-6 months
BMIHT,8726,98.49,Standing Height Comment,2-150 years
BMIARML,8660,97.74,Upper Arm Length Comment,2 months-150 years
BMIARMC,8655,97.69,Arm Circumference Comment,2 months-150 years
BMIWT,8515,96.11,Weight Comment,0-150 years
BMIWAIST,8513,96.08,Waist Circumference Comment,2-150 years
BMIHIP,8499,95.93,Hip Circumference Comment,12-150 years
BMILEG,8464,95.53,Upper Leg Length Comment,8-150 years


The columns fall into two groups. `BMX…` columns hold measurements. `BMI…` columns are *comment*
columns that are filled in only when the examiner recorded a comment, so they are empty for most
participants by design. The next cell reports missing values separately for the two groups.

In [14]:
bmx_measure_cols = [c for c in bmx.columns if c.startswith("BMX")]
bmx_comment_cols = [c for c in bmx.columns if c.startswith("BMI")]

for group_name, cols in [("Measurement columns (BMX...)", bmx_measure_cols),
                         ("Comment columns (BMI...)", bmx_comment_cols)]:
    n_missing = int(bmx[cols].isna().sum().sum())
    print(f"{group_name}: {n_missing:,} missing of {bmx[cols].size:,} cells "
          f"({n_missing / bmx[cols].size * 100:.1f}%)")

Measurement columns (BMX...): 22,921 missing of 88,600 cells (25.9%)
Comment columns (BMI...): 77,734 missing of 79,740 cells (97.5%)


## Step 4: Incomplete data

`BMDSTATS` is the documented examination status code. It identifies participants whose body
measures examination was only partly completed or not completed.

In [15]:
# Value labels from the BMX_L codebook
BMDSTATS_LABELS = {
    1: "Complete data for age group",
    2: "Partial: Only height and weight obtained",
    3: "Other partial exam",
    4: "No body measures exam data",
}
bmx_status = bmx["BMDSTATS"].value_counts(dropna=False).sort_index().to_frame("count")
bmx_status["documented meaning"] = bmx_status.index.map(BMDSTATS_LABELS)
bmx_status["pct_of_rows"] = (bmx_status["count"] / len(bmx) * 100).round(2)
bmx_status

,count,documented meaning,pct_of_rows
1.0,8235,Complete data for age group,92.95
2.0,147,Partial: Only height and weight obtained,1.66
3.0,409,Other partial exam,4.62
4.0,69,No body measures exam data,0.78


The comment columns use code `1` = *Could not obtain*. This is a documented "unavailable" code: the
measurement was attempted or due but no value was obtained. Other comment codes (for example
`BMIWT = 3` *Clothing*, `BMIWT = 4` *Medical appliance*, `BMIHT = 3` *Not straight*) qualify a
measurement that **was** taken, so they are not counted as unavailable.

In [16]:
BMX_CODES = {col: {1: "Could not obtain"} for col in bmx_comment_cols}
bmx_code_table = special_code_table(bmx, BMX_CODES)
bmx_code_table

,column,code,documented meaning,count,pct_of_rows
0,BMIWT,1,Could not obtain,30,0.34
1,BMIRECUM,1,Could not obtain,18,0.20
2,BMIHEAD,1,Could not obtain,0,0.00
3,BMIHT,1,Could not obtain,46,0.52
4,BMILEG,1,Could not obtain,396,4.47
5,BMIARML,1,Could not obtain,200,2.26
6,BMIARMC,1,Could not obtain,205,2.31
7,BMIWAIST,1,Could not obtain,347,3.92
8,BMIHIP,1,Could not obtain,361,4.07


The next cell counts records without weight, standing height or BMI. Standing height and BMI are
only targeted at ages 2 and over, so this count includes younger children for whom the blank is
expected.

In [17]:
# Records missing the fields most often needed for analysis
bmx_key_cols = ["BMXWT", "BMXHT", "BMXBMI"]
bmx_key_incomplete = int(bmx[bmx_key_cols].isna().any(axis=1).sum())
bmx_not_complete = int((bmx["BMDSTATS"] != 1).sum())
bmx_could_not_obtain = int(bmx_code_table["count"].sum())

print(f"Records with status other than 'Complete data for age group': {bmx_not_complete:,}")
print(f"Records missing weight, standing height or BMI              : {bmx_key_incomplete:,}")
print(f"'Could not obtain' comments across all measurements         : {bmx_could_not_obtain:,}")

Records with status other than 'Complete data for age group': 625
Records missing weight, standing height or BMI              : 389
'Could not obtain' comments across all measurements         : 1,603


**Legitimate values that may look unusual.** The file covers all ages from birth, so very low
weights and lengths are expected. The observed ranges are shown for reference only; no value is
flagged or changed.

In [18]:
bmx[bmx_measure_cols].agg(["min", "max"]).T

,min,max
BMXWT,2.7,248.2
BMXRECUM,48.5,118.8
BMXHEAD,34.4,46.5
BMXHT,79.1,200.7
BMXBMI,11.1,74.8
BMXLEG,24.9,51.6
BMXARML,10.0,49.2
BMXARMC,12.0,63.3
BMXWAIST,39.8,187.0
BMXHIP,69.9,187.1


## Step 5: Duplicate data

`SEQN` is the respondent sequence number and is expected to be unique in this file (one row per
participant).

In [19]:
bmx_duplicates = duplicate_overview(bmx, id_columns=["SEQN"])

Exact duplicate rows: 0 (0.0% of rows)
Rows repeating an earlier identifier ['SEQN']: 0


## Step 6: Summary of findings

In [20]:
record_results(BMX_NAME, bmx, bmx_missing, bmx_duplicates, [
    f"Missing: {bmx_missing['total_missing']:,} cells ({bmx_missing['missing_pct']}%), in "
    f"{bmx_missing['columns_with_missing']} of {bmx.shape[1]} columns; most are in comment columns "
    f"and in measurements with a documented age-restricted target.",
    f"Incomplete: {bmx_not_complete:,} records have an exam status other than complete; "
    f"{bmx_could_not_obtain:,} 'Could not obtain' codes; {bmx_key_incomplete:,} records lack weight, height or BMI.",
    f"Duplicates: {bmx_duplicates['exact_duplicates']} exact duplicate rows, "
    f"{bmx_duplicates['id_duplicates']} repeated SEQN.",
    "Limitation: the file has no age variable, so age-ineligible blanks cannot be separated from "
    "unobtained measurements without the NHANES demographics file.",
])

Summary - NHANES Body Measures (BMX_L, 2021-2023)
 - Missing: 107,023 cells (54.91%), in 20 of 22 columns; most are in comment columns and in measurements with a documented age-restricted target.
 - Incomplete: 625 records have an exam status other than complete; 1,603 'Could not obtain' codes; 389 records lack weight, height or BMI.
 - Duplicates: 0 exact duplicate rows, 0 repeated SEQN.
 - Limitation: the file has no age variable, so age-ineligible blanks cannot be separated from unobtained measurements without the NHANES demographics file.


---
# Dataset 2 — NHANES Dietary Interview, Total Nutrient Intakes, First Day (`DR1TOT_L`, 2021–2023)

- **Publisher:** CDC / National Center for Health Statistics
- **Documentation and codebook:** <https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1TOT_L.htm>
- **Data file:** `DR1TOT_L.xpt` from the same CDC page
- **Unit of observation:** one row per participant (`SEQN`), with that participant's total
  nutrient intake on the first 24-hour dietary recall day

## Step 1: Load the dataset

In [21]:
DR1_NAME = "NHANES Dietary Interview - Total Nutrient Intakes, First Day (DR1TOT_L, 2021-2023)"
DR1_SOURCE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1TOT_L.htm"
DR1_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1TOT_L.xpt"

dr1_path = download_file(
    DR1_URL, DATA_DIR / "DR1TOT_L.xpt",
    manual_hint=f"open {DR1_SOURCE}, download the XPT data file and save it as data/DR1TOT_L.xpt",
)
dr1, dr1_meta = pyreadstat.read_xport(str(dr1_path))
dr1_labels = dr1_meta.column_names_to_labels

show_dimensions(dr1, DR1_NAME, DR1_SOURCE)

Using cached file: data/DR1TOT_L.xpt (11.9 MB)
Dataset : NHANES Dietary Interview - Total Nutrient Intakes, First Day (DR1TOT_L, 2021-2023)
Source  : https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DR1TOT_L.htm
Rows    : 8,860
Columns : 168


In [22]:
show_columns(dr1)

Data types: {'float64': 168}

All 168 column names:
SEQN (float64), WTDRD1 (float64), WTDR2D (float64), DR1DRSTZ (float64), DR1EXMER (float64), DRABF (float64), DRDINT (float64), DR1DBIH (float64), DR1DAY (float64), DR1LANG (float64), DR1MRESP (float64), DR1HELP (float64), DBQ095Z (float64), DBD100 (float64), DRQSPREP (float64), DR1STY (float64), DR1SKY (float64), DRQSDIET (float64), DRQSDT1 (float64), DRQSDT2 (float64), DRQSDT3 (float64), DRQSDT4 (float64), DRQSDT5 (float64), DRQSDT6 (float64), DRQSDT7 (float64), DRQSDT8 (float64), DRQSDT9 (float64), DRQSDT10 (float64), DRQSDT11 (float64), DRQSDT12 (float64), DRQSDT91 (float64), DR1TNUMF (float64), DR1TKCAL (float64), DR1TPROT (float64), DR1TCARB (float64), DR1TSUGR (float64), DR1TFIBE (float64), DR1TTFAT (float64), DR1TSFAT (float64), DR1TMFAT (float64), DR1TPFAT (float64), DR1TCHOL (float64), DR1TATOC (float64), DR1TATOA (float64), DR1TRET (float64), DR1TVARA (float64), DR1TACAR (float64), DR1TBCAR (float64), DR1TCRYP (float64), DR1

## Step 2: Representative excerpt

The dataset has 168 columns, so the excerpt shows only a subset: the identifier, the day-one sample
weight, the recall status and a few of the main nutrient totals. The labels of the selected columns
are listed first.

In [23]:
dr1_excerpt_cols = ["SEQN", "WTDRD1", "DR1DRSTZ", "DR1DAY", "DRQSDIET", "DR1TNUMF",
                    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR", "DR1TTFAT", "DR1TSODI"]
pd.Series({col: dr1_labels[col] for col in dr1_excerpt_cols}, name="label").to_frame()

,label
SEQN,Respondent sequence number
WTDRD1,Dietary day one sample weight
DR1DRSTZ,Dietary recall status
DR1DAY,Intake day of the week
DRQSDIET,On special diet?
DR1TNUMF,Number of foods/beverages reported
DR1TKCAL,Energy (kcal)
DR1TPROT,Protein (gm)
DR1TCARB,Carbohydrate (gm)
DR1TSUGR,Total sugars (gm)


In [24]:
show_excerpt(dr1, dr1_excerpt_cols)

,SEQN,WTDRD1,DR1DRSTZ,DR1DAY,DRQSDIET,DR1TNUMF,DR1TKCAL,DR1TPROT,DR1TCARB,DR1TSUGR,DR1TTFAT,DR1TSODI
222,130673.0,0.000000,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3859,135587.0,65792.054213,1.0,3.0,2.0,23.0,2181.0,73.32,226.97,100.97,94.82,4026.0
4011,135797.0,32421.938542,1.0,1.0,2.0,17.0,2146.0,70.62,223.69,60.04,109.57,4312.0
4890,136987.0,20370.752382,1.0,5.0,2.0,10.0,882.0,18.79,117.93,51.38,38.08,817.0
5952,138407.0,26322.692611,1.0,5.0,2.0,11.0,2176.0,103.80,211.99,62.16,102.34,3503.0
6280,138843.0,21794.896298,1.0,3.0,2.0,9.0,1743.0,58.07,223.60,62.03,67.07,2249.0
6347,138936.0,0.000000,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7768,140845.0,43579.455381,1.0,4.0,2.0,11.0,1059.0,38.04,117.46,47.25,49.08,1990.0


## Step 3: Missing data

In [25]:
dr1_missing = missing_overview(dr1)

Total cells                    : 1,488,480
Total missing values           : 739,930 (49.71% of all cells)
Columns with missing values    : 165 of 168
Columns that are entirely empty: 0


With 168 columns the full table is too long to print. The next cells show the missing-value figures
for the excerpt columns and a count of columns by level of missingness.

In [26]:
dr1_missing_table = missing_table(dr1)
dr1_missing_table["label"] = dr1_missing_table.index.map(dr1_labels)
dr1_missing_table.loc[dr1_excerpt_cols]

,missing_count,missing_pct,label
SEQN,0,0.00,Respondent sequence number
WTDRD1,0,0.00,Dietary day one sample weight
DR1DRSTZ,0,0.00,Dietary recall status
DR1DAY,2063,23.28,Intake day of the week
DRQSDIET,2063,23.28,On special diet?
DR1TNUMF,2106,23.77,Number of foods/beverages reported
DR1TKCAL,2166,24.45,Energy (kcal)
DR1TPROT,2166,24.45,Protein (gm)
DR1TCARB,2166,24.45,Carbohydrate (gm)
DR1TSUGR,2166,24.45,Total sugars (gm)


In [27]:
# How many columns fall into each missing-percentage band
bands = pd.cut(dr1_missing_table["missing_pct"],
               bins=[-0.01, 0, 25, 50, 75, 100],
               labels=["0%", ">0-25%", ">25-50%", ">50-75%", ">75-100%"])
bands.value_counts().sort_index().to_frame("number of columns")

,number of columns
0%,3
>0-25%,85
>25-50%,2
>50-75%,34
>75-100%,44


The nutrient totals (`DR1T…` columns) are only released for participants with a usable recall. The
cross-tabulation below checks this against the documented recall status `DR1DRSTZ`
(1 = *Reliable and met the minimum criteria*, 2 = *Not reliable or not met the minimum criteria*,
4 = *Reported consuming breast-milk*, 5 = *Not done*).

In [28]:
dr1_nutrient_cols = [c for c in dr1.columns if c.startswith("DR1T") and c not in ("DR1TNUMF", "DR1TWSZ")]
print(f"Nutrient-total columns: {len(dr1_nutrient_cols)}")
print("Distinct missing counts across these columns:",
      sorted(dr1[dr1_nutrient_cols].isna().sum().unique()))

pd.crosstab(dr1["DR1DRSTZ"], dr1["DR1TKCAL"].isna().map({True: "energy missing", False: "energy present"}))

Nutrient-total columns: 65
Distinct missing counts across these columns: [2166]


DR1TKCAL,energy missing,energy present
DR1DRSTZ,,
1.0,0,6694
2.0,43,0
4.0,60,0
5.0,2063,0


The many columns in the >75% band are follow-up questions that are only asked after a specific
earlier answer (for example the type of special diet, or how often each kind of fish was eaten).
A blank there is a documented skip, not a lost value.

## Step 4: Incomplete data

Three documented conventions mark incomplete information in this file without always appearing as
missing values in pandas:

- `DR1DRSTZ` values other than 1 (recall not reliable, breast-fed infant, or not done);
- `WTDRD1 = 0`, documented as *Day 1 dietary recall not done/incomplete*;
- *Refused* (7 / 77) and *Don't know* (9 / 99) codes on questionnaire items.

In [29]:
DR1DRSTZ_LABELS = {
    1: "Reliable and met the minimum criteria",
    2: "Not reliable or not met the minimum criteria",
    4: "Reported consuming breast-milk",
    5: "Not done",
}
dr1_status = dr1["DR1DRSTZ"].value_counts(dropna=False).sort_index().to_frame("count")
dr1_status["documented meaning"] = dr1_status.index.map(DR1DRSTZ_LABELS)
dr1_status["pct_of_rows"] = (dr1_status["count"] / len(dr1) * 100).round(2)
dr1_status

,count,documented meaning,pct_of_rows
1.0,6694,Reliable and met the minimum criteria,75.55
2.0,43,Not reliable or not met the minimum criteria,0.49
4.0,60,Reported consuming breast-milk,0.68
5.0,2063,Not done,23.28


In [30]:
# Special codes transcribed from the DR1TOT_L codebook (only codes listed there are included)
DR1_CODES = {
    "WTDRD1":   {0: "Day 1 dietary recall not done/incomplete"},
    "DR1MRESP": {77: "Refused", 99: "Don't know"},
    "DR1HELP":  {77: "Refused", 99: "Don't know"},
    "DBQ095Z":  {99: "Don't know"},
    "DBD100":   {7: "Refused", 9: "Don't know"},
    "DRQSPREP": {9: "Don't know"},
    "DR1STY":   {9: "Don't know"},
    "DR1SKY":   {99: "Don't know"},
    "DRQSDIET": {9: "Don't know"},
    "DR1_300":  {7: "Refused", 9: "Don't know"},
    "DR1TWSZ":  {99: "Don't know"},
    "DRD340":   {7: "Refused", 9: "Don't know"},
    "DRD360":   {7: "Refused", 9: "Don't know"},
}
dr1_code_table = special_code_table(dr1, DR1_CODES)
dr1_code_table

,column,code,documented meaning,count,pct_of_rows
0,WTDRD1,0,Day 1 dietary recall not done/incomplete,2106,23.77
1,DR1MRESP,77,Refused,0,0.00
2,DR1MRESP,99,Don't know,0,0.00
3,DR1HELP,77,Refused,0,0.00
4,DR1HELP,99,Don't know,6,0.07
5,DBQ095Z,99,Don't know,53,0.60
6,DBD100,7,Refused,0,0.00
7,DBD100,9,Don't know,10,0.11
8,DRQSPREP,9,Don't know,87,0.98
9,DR1STY,9,Don't know,45,0.51


**The same number is not always a nonresponse code.** The codes above were checked column by
column. In other columns of this file, 7 and 9 are ordinary answers, for example `DR1DAY = 7`
(*Saturday*) and `DRQSDT9 = 9` (*Low carbohydrate diet*). They are counted below only to show that
they exist; they are valid values.

In [31]:
print("DR1DAY == 7 (Saturday)                :", int((dr1["DR1DAY"] == 7).sum()))
print("DRQSDT9 == 9 (Low carbohydrate diet)  :", int((dr1["DRQSDT9"] == 9).sum()))

DR1DAY == 7 (Saturday)                : 550
DRQSDT9 == 9 (Low carbohydrate diet)  : 70


In [32]:
dr1_not_reliable = int((dr1["DR1DRSTZ"] != 1).sum())
dr1_zero_weight = int((dr1["WTDRD1"] == 0).sum())
dr1_no_energy = int(dr1["DR1TKCAL"].isna().sum())
dr1_dk_refused = int(dr1_code_table.loc[dr1_code_table["column"] != "WTDRD1", "count"].sum())

print(f"Records with recall status other than 'Reliable and met the minimum criteria': {dr1_not_reliable:,}")
print(f"Records with day-one weight WTDRD1 = 0 (not done/incomplete)                : {dr1_zero_weight:,}")
print(f"Records without a total energy value (DR1TKCAL)                             : {dr1_no_energy:,}")
print(f"'Refused' / 'Don't know' codes in the checked questionnaire columns          : {dr1_dk_refused:,}")

Records with recall status other than 'Reliable and met the minimum criteria': 2,166
Records with day-one weight WTDRD1 = 0 (not done/incomplete)                : 2,106
Records without a total energy value (DR1TKCAL)                             : 2,166
'Refused' / 'Don't know' codes in the checked questionnaire columns          : 627


## Step 5: Duplicate data

This is the *totals* file, with one row per participant, so `SEQN` is expected to be unique.
(The separate NHANES *individual foods* files have many rows per participant by design; they are
not part of this investigation.)

In [33]:
dr1_duplicates = duplicate_overview(dr1, id_columns=["SEQN"])

Exact duplicate rows: 0 (0.0% of rows)
Rows repeating an earlier identifier ['SEQN']: 0


## Step 6: Summary of findings

In [34]:
record_results(DR1_NAME, dr1, dr1_missing, dr1_duplicates, [
    f"Missing: {dr1_missing['total_missing']:,} cells ({dr1_missing['missing_pct']}%), in "
    f"{dr1_missing['columns_with_missing']} of {dr1.shape[1]} columns; nutrient totals are blank for "
    f"{dr1_no_energy:,} records, and many follow-up questions are blank through documented skips.",
    f"Incomplete: {dr1_not_reliable:,} records have a recall status other than reliable "
    f"({dr1_zero_weight:,} have day-one weight 0 = not done/incomplete); "
    f"{dr1_dk_refused:,} 'Refused'/'Don't know' codes in the checked columns.",
    f"Duplicates: {dr1_duplicates['exact_duplicates']} exact duplicate rows, "
    f"{dr1_duplicates['id_duplicates']} repeated SEQN.",
    "Limitation: special codes were checked for the questionnaire columns listed above, not for all 168 columns.",
])

Summary - NHANES Dietary Interview - Total Nutrient Intakes, First Day (DR1TOT_L, 2021-2023)
 - Missing: 739,930 cells (49.71%), in 165 of 168 columns; nutrient totals are blank for 2,166 records, and many follow-up questions are blank through documented skips.
 - Incomplete: 2,166 records have a recall status other than reliable (2,106 have day-one weight 0 = not done/incomplete); 627 'Refused'/'Don't know' codes in the checked columns.
 - Duplicates: 0 exact duplicate rows, 0 repeated SEQN.
 - Limitation: special codes were checked for the questionnaire columns listed above, not for all 168 columns.


---
# Dataset 3 — CDC BRFSS 2024

- **Publisher:** Centers for Disease Control and Prevention
- **Official release page:** <https://www.cdc.gov/brfss/annual_data/annual_2024.html>
- **Data file:** `LLCP2024XPT.zip` (2024 BRFSS data, SAS transport format) linked from that page
- **Codebook:** "2024 BRFSS Codebook" (`codebook24_llcp-v2-508.zip`) linked from the same page
- **Unit of observation:** one row per completed or partially completed telephone interview

## Step 1: Load the dataset

The zip file (about 83 MB) contains a single transport file of about 1.1 GB. It is extracted once
into `data/` and then read in full with `pyreadstat` (roughly 1.2 GB of memory). The **complete**
file is analysed; nothing below is based on a sample.

In [35]:
BRFSS_NAME = "CDC BRFSS 2024"
BRFSS_SOURCE = "https://www.cdc.gov/brfss/annual_data/annual_2024.html"
BRFSS_URL = "https://www.cdc.gov/brfss/annual_data/2024/files/LLCP2024XPT.zip"

brfss_zip = download_file(
    BRFSS_URL, DATA_DIR / "LLCP2024XPT.zip",
    manual_hint=f"open {BRFSS_SOURCE}, download '2024 BRFSS Data (SAS Transport Format)' "
                "and save it as data/LLCP2024XPT.zip",
)

# Extract the single transport file (its name inside the zip ends with a space, so it is renamed)
brfss_xpt = DATA_DIR / "LLCP2024.XPT"
if not brfss_xpt.exists():
    with zipfile.ZipFile(brfss_zip) as archive:
        members = archive.namelist()
        assert len(members) == 1, f"Expected one file in the zip, found: {members}"
        with archive.open(members[0]) as source, open(brfss_xpt, "wb") as target:
            while True:
                block = source.read(16 * 1024 * 1024)
                if not block:
                    break
                target.write(block)
print(f"Transport file: {brfss_xpt} ({brfss_xpt.stat().st_size / 1e6:,.0f} MB)")

Using cached file: data/LLCP2024XPT.zip (83.0 MB)
Transport file: data/LLCP2024.XPT (1,094 MB)


In [36]:
# The variable labels in this file are Windows-1252 encoded
brfss, brfss_meta = pyreadstat.read_xport(str(brfss_xpt), encoding="cp1252")
brfss_labels = brfss_meta.column_names_to_labels

show_dimensions(brfss, BRFSS_NAME, BRFSS_SOURCE)
print(f"Memory  : {brfss.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

Dataset : CDC BRFSS 2024
Source  : https://www.cdc.gov/brfss/annual_data/annual_2024.html
Rows    : 457,670
Columns : 301
Memory  : 1,226 MB


In [37]:
show_columns(brfss)

Data types: {'float64': 296, 'object': 5}

All 301 column names:
_STATE (float64), FMONTH (float64), IDATE (object), IMONTH (object), IDAY (object), IYEAR (object), DISPCODE (float64), SEQNO (object), _PSU (float64), CTELENM1 (float64), PVTRESD1 (float64), COLGHOUS (float64), STATERE1 (float64), CELPHON1 (float64), LADULT1 (float64), NUMADULT (float64), RESPSLC1 (float64), LANDSEX3 (float64), SAFETIME (float64), CTELNUM1 (float64), CELLFON5 (float64), CADULT1 (float64), CELLSEX3 (float64), PVTRESD3 (float64), CCLGHOUS (float64), CSTATE1 (float64), LANDLINE (float64), HHADULT (float64), SEXVAR (float64), GENHLTH (float64), PHYSHLTH (float64), MENTHLTH (float64), POORHLTH (float64), PRIMINS2 (float64), PERSDOC3 (float64), MEDCOST1 (float64), CHECKUP1 (float64), EXERANY2 (float64), LASTDEN4 (float64), RMVTETH4 (float64), CVDINFR4 (float64), CVDCRHD4 (float64), CVDSTRK3 (float64), ASTHMA3 (float64), ASTHNOW (float64), CHCSCNC1 (float64), CHCOCNC1 (float64), CHCCOPD3 (float64), ADDEPEV3 (fl

## Step 2: Representative excerpt

The dataset has 301 columns. The excerpt shows a small subset: record identifiers, interview
disposition, and a few core health and demographic questions. Values are shown exactly as stored,
so coded answers (for example `88` or `99`) appear as numbers; their meaning is covered in Step 4.

In [38]:
brfss_excerpt_cols = ["_STATE", "SEQNO", "IDATE", "DISPCODE", "SEXVAR", "_AGE80", "GENHLTH",
                      "PHYSHLTH", "MENTHLTH", "EDUCA", "INCOME3", "WEIGHT2", "HEIGHT3", "_BMI5"]
pd.Series({col: brfss_labels[col] for col in brfss_excerpt_cols}, name="label").to_frame()

,label
_STATE,STATE FIPS CODE
SEQNO,ANNUAL SEQUENCE NUMBER
IDATE,INTERVIEW DATE
DISPCODE,FINAL DISPOSITION
SEXVAR,SEX OF RESPONDENT
_AGE80,IMPUTED AGE VALUE COLLAPSED ABOVE 80
GENHLTH,GENERAL HEALTH
PHYSHLTH,NUMBER OF DAYS PHYSICAL HEALTH NOT GOOD
MENTHLTH,NUMBER OF DAYS MENTAL HEALTH NOT GOOD
EDUCA,EDUCATION LEVEL


In [39]:
show_excerpt(brfss, brfss_excerpt_cols)

,_STATE,SEQNO,IDATE,DISPCODE,SEXVAR,_AGE80,GENHLTH,PHYSHLTH,MENTHLTH,EDUCA,INCOME3,WEIGHT2,HEIGHT3,_BMI5
14182,4.0,2024002959,07252024,1100.0,1.0,27.0,2.0,88.0,5.0,6.0,6.0,190.0,600.0,2577.0
276632,36.0,2024015634,11222024,1100.0,2.0,53.0,3.0,10.0,15.0,5.0,6.0,9999.0,506.0,NaN
348161,45.0,2024001212,05052024,1100.0,2.0,62.0,2.0,30.0,1.0,3.0,10.0,110.0,504.0,1888.0
349771,45.0,2024000152,06292024,1200.0,1.0,67.0,5.0,30.0,88.0,3.0,1.0,250.0,507.0,3916.0
376630,49.0,2024002018,10152024,1100.0,1.0,34.0,3.0,3.0,15.0,2.0,4.0,210.0,603.0,2625.0
421533,53.0,2024020497,11072024,1100.0,2.0,40.0,2.0,3.0,15.0,6.0,8.0,166.0,504.0,2849.0
424413,53.0,2024023377,01022025,1100.0,2.0,61.0,1.0,88.0,88.0,4.0,11.0,150.0,508.0,2281.0
428211,54.0,2024001525,11072024,1100.0,1.0,66.0,2.0,88.0,88.0,6.0,6.0,165.0,506.0,2663.0


## Step 3: Missing data

In [40]:
brfss_missing = missing_overview(brfss)

Total cells                    : 137,758,670
Total missing values           : 66,613,951 (48.36% of all cells)
Columns with missing values    : 249 of 301
Columns that are entirely empty: 2


In [41]:
# How many columns fall into each missing-percentage band
brfss_missing_table = missing_table(brfss)
brfss_missing_table["label"] = brfss_missing_table.index.map(brfss_labels)

bands = pd.cut(brfss_missing_table["missing_pct"],
               bins=[-0.01, 0, 25, 50, 75, 100],
               labels=["0%", ">0-25%", ">25-50%", ">50-75%", ">75-100%"])
bands.value_counts().sort_index().to_frame("number of columns")

,number of columns
0%,80
>0-25%,44
>25-50%,7
>50-75%,57
>75-100%,113


In [42]:
# Missing values in the excerpt columns
brfss_missing_table.loc[brfss_excerpt_cols]

,missing_count,missing_pct,label
_STATE,0,0.00,STATE FIPS CODE
SEQNO,0,0.00,ANNUAL SEQUENCE NUMBER
IDATE,0,0.00,INTERVIEW DATE
DISPCODE,0,0.00,FINAL DISPOSITION
SEXVAR,0,0.00,SEX OF RESPONDENT
_AGE80,0,0.00,IMPUTED AGE VALUE COLLAPSED ABOVE 80
GENHLTH,5,0.00,GENERAL HEALTH
PHYSHLTH,5,0.00,NUMBER OF DAYS PHYSICAL HEALTH NOT GOOD
MENTHLTH,3,0.00,NUMBER OF DAYS MENTAL HEALTH NOT GOOD
EDUCA,7,0.00,EDUCATION LEVEL


In [43]:
# The ten columns with the most missing values
brfss_missing_table.head(10)

,missing_count,missing_pct,label
RCSGEND1,457670,100.00,SEX OF CHILD
RCSXBRTH,457670,100.00,CHILD´S SEX AT BIRT
COLGHOUS,457656,100.00,DO YOU LIVE IN COLLEGE HOUSING?
CSRVCTL2,457088,99.87,IS PAIN UNDER CONTROL?
ICFQSTVR,456931,99.84,None
CCLGHOUS,456244,99.69,DO YOU LIVE IN COLLEGE HOUSING?
HPVDSHT,455650,99.56,None
CSRVINST,455045,99.43,INSTRUCTIONS WRITTEN OR PRINTED
NOBCUSE8,454899,99.39,WHAT WAS MAIN REASON FOR NOT DOING ANYTH
LASTSIG4,454443,99.29,TIME SINCE LAST SIGMOIDOSCOPY/COLONOSCOP


In the BRFSS codebook a blank is labelled **"Not asked or Missing"**. A blank therefore covers
several documented situations: questions skipped because of an earlier answer, optional modules
that a state did not use, landline-only or cell-only questions, and questions not reached in a
partially completed interview. The file does not distinguish between these.

One of these situations can be checked directly: `DISPCODE` separates completed interviews (1100)
from partially completed ones (1200).

In [44]:
# Average share of blank cells per record, by interview disposition
brfss_row_missing_pct = brfss.isna().mean(axis=1) * 100
(brfss_row_missing_pct.groupby(brfss["DISPCODE"]).agg(["count", "mean"])
 .rename(columns={"count": "records", "mean": "avg % of columns blank"}).round(2))

,records,avg % of columns blank
DISPCODE,,
1100.0,372224,46.67
1200.0,85446,55.69


## Step 4: Incomplete data

BRFSS records nonresponse with numeric codes, which pandas does **not** treat as missing. The codes
differ by the width of the field (7/9, 77/99, 7777/9999), and some columns have no "Don't know"
option. The codes below were transcribed from the 2024 codebook for a set of commonly analysed
variables.

In [45]:
DK = "Don't know/Not sure"
BRFSS_CODES = {
    "GENHLTH":  {7: DK, 9: "Refused"},
    "PHYSHLTH": {77: DK, 99: "Refused"},
    "MENTHLTH": {77: DK, 99: "Refused"},
    "EXERANY2": {7: DK, 9: "Refused"},
    "DIABETE4": {7: DK, 9: "Refused"},
    "SMOKE100": {7: DK, 9: "Refused"},
    "MARITAL":  {9: "Refused"},
    "EDUCA":    {9: "Refused"},
    "EMPLOY1":  {9: "Refused"},
    "CHILDREN": {99: "Refused"},
    "INCOME3":  {77: DK, 99: "Refused"},
    "WEIGHT2":  {7777: DK, 9999: "Refused"},
    "HEIGHT3":  {7777: DK, 9999: "Refused"},
}
brfss_code_table = special_code_table(brfss, BRFSS_CODES)
brfss_code_table

,column,code,documented meaning,count,pct_of_rows
0,GENHLTH,7,Don't know/Not sure,915,0.20
1,GENHLTH,9,Refused,390,0.09
2,PHYSHLTH,77,Don't know/Not sure,8765,1.92
3,PHYSHLTH,99,Refused,2297,0.50
4,MENTHLTH,77,Don't know/Not sure,5594,1.22
5,MENTHLTH,99,Refused,2559,0.56
6,EXERANY2,7,Don't know/Not sure,860,0.19
7,EXERANY2,9,Refused,452,0.10
8,DIABETE4,7,Don't know/Not sure,798,0.17
9,DIABETE4,9,Refused,232,0.05


For each of these variables, the next table separates the three situations: an actual blank, a
documented nonresponse code, and a substantive answer.

In [46]:
rows = []
for column, code_meanings in BRFSS_CODES.items():
    blank = int(brfss[column].isna().sum())
    coded = int(brfss[column].isin(list(code_meanings)).sum())
    rows.append({
        "column": column,
        "label": brfss_labels[column],
        "blank (not asked or missing)": blank,
        "nonresponse code": coded,
        "substantive answer": len(brfss) - blank - coded,
        "blank or nonresponse (%)": round((blank + coded) / len(brfss) * 100, 2),
    })
brfss_incomplete_table = pd.DataFrame(rows).set_index("column")
brfss_incomplete_table

,label,blank (not asked or missing),nonresponse code,substantive answer,blank or nonresponse (%)
column,,,,,
GENHLTH,GENERAL HEALTH,5,1305,456360,0.29
PHYSHLTH,NUMBER OF DAYS PHYSICAL HEALTH NOT GOOD,5,11062,446603,2.42
MENTHLTH,NUMBER OF DAYS MENTAL HEALTH NOT GOOD,3,8153,449514,1.78
EXERANY2,EXERCISE IN PAST 30 DAYS,3,1312,456355,0.29
DIABETE4,(EVER TOLD) YOU HAD DIABETES,4,1030,456636,0.23
SMOKE100,SMOKED AT LEAST 100 CIGARETTES,28860,2612,426198,6.88
MARITAL,MARITAL STATUS,9,4213,453448,0.92
EDUCA,EDUCATION LEVEL,7,2356,455307,0.52
EMPLOY1,EMPLOYMENT STATUS,3303,5099,449268,1.84


In [47]:
# Records with a blank or a nonresponse code in at least one of the variables above
brfss_key_cols = list(BRFSS_CODES)
is_blank = brfss[brfss_key_cols].isna()
is_coded = pd.DataFrame({col: brfss[col].isin(list(codes)) for col, codes in BRFSS_CODES.items()})

brfss_key_incomplete = int((is_blank | is_coded).any(axis=1).sum())
brfss_partial = int((brfss["DISPCODE"] == 1200).sum())
brfss_nonresponse_codes = int(brfss_code_table["count"].sum())

print(f"Partially completed interviews (DISPCODE = 1200): {brfss_partial:,} "
      f"({brfss_partial / len(brfss) * 100:.2f}%)")
print(f"Nonresponse codes in the {len(brfss_key_cols)} checked variables: {brfss_nonresponse_codes:,}")
print(f"Records with a blank or nonresponse code in at least one of them: {brfss_key_incomplete:,} "
      f"({brfss_key_incomplete / len(brfss) * 100:.2f}%)")

Partially completed interviews (DISPCODE = 1200): 85,446 (18.67%)
Nonresponse codes in the 13 checked variables: 153,750
Records with a blank or nonresponse code in at least one of them: 129,571 (28.31%)


**Calculated variables hide nonresponse differently.** CDC's calculated variables (names starting
with `_`) fold blanks and nonresponse into their own code or into a blank. For example, the
codebook defines `_INCOMG1 = 9` as *Don't know/Not sure/Missing* and leaves `_BMI5` blank when
weight or height is unknown or refused.

In [48]:
print("_INCOMG1 == 9 (Don't know/Not sure/Missing):", f"{int((brfss['_INCOMG1'] == 9).sum()):,}")
print("_BMI5 blank (Don't know/Refused/Missing)   :", f"{int(brfss['_BMI5'].isna().sum()):,}")

_INCOMG1 == 9 (Don't know/Not sure/Missing): 87,423
_BMI5 blank (Don't know/Refused/Missing)   : 43,037


**Legitimate values that may look unusual.** The following are valid, documented answers and are
*not* nonresponse:

- `88` = *None* in `PHYSHLTH`, `MENTHLTH` and `CHILDREN`;
- `WEIGHT2` values 9023–9352 and `HEIGHT3` values 9061–9998 are answers given in metric units (the
  leading 9 marks a metric value);
- `_BMI5` has two implied decimal places (2583 means 25.83).

In [49]:
print("PHYSHLTH == 88 (None)            :", f"{int((brfss['PHYSHLTH'] == 88).sum()):,}")
print("MENTHLTH == 88 (None)            :", f"{int((brfss['MENTHLTH'] == 88).sum()):,}")
print("CHILDREN == 88 (None)            :", f"{int((brfss['CHILDREN'] == 88).sum()):,}")
print("WEIGHT2 metric (9023 to 9352)    :", f"{int(brfss['WEIGHT2'].between(9023, 9352).sum()):,}")
print("HEIGHT3 metric (9061 to 9998)    :", f"{int(brfss['HEIGHT3'].between(9061, 9998).sum()):,}")

PHYSHLTH == 88 (None)            : 267,576
MENTHLTH == 88 (None)            : 269,909
CHILDREN == 88 (None)            : 336,299
WEIGHT2 metric (9023 to 9352)    : 4,190
HEIGHT3 metric (9061 to 9998)    : 7,793


## Step 5: Duplicate data

BRFSS has no single-column record identifier. `SEQNO` is an annual sequence number that restarts
in each state, so the same `SEQNO` legitimately appears in many states. The appropriate identifier
is the combination of `_STATE` and `SEQNO`.

In [50]:
brfss_duplicates = duplicate_overview(brfss, id_columns=["_STATE", "SEQNO"])

# Shown for contrast: SEQNO alone repeats across states by design
print(f"Rows repeating an earlier SEQNO when the state is ignored: {int(brfss['SEQNO'].duplicated().sum()):,}")

Exact duplicate rows: 0 (0.0% of rows)
Rows repeating an earlier identifier ['_STATE', 'SEQNO']: 0
Rows repeating an earlier SEQNO when the state is ignored: 413,757


## Step 6: Summary of findings

In [51]:
record_results(BRFSS_NAME, brfss, brfss_missing, brfss_duplicates, [
    f"Missing: {brfss_missing['total_missing']:,} cells ({brfss_missing['missing_pct']}%), in "
    f"{brfss_missing['columns_with_missing']} of {brfss.shape[1]} columns "
    f"({brfss_missing['columns_all_missing']} columns entirely blank); blanks are documented as 'Not asked or Missing'.",
    f"Incomplete: {brfss_partial:,} partially completed interviews; {brfss_nonresponse_codes:,} "
    f"'Don't know'/'Refused' codes in {len(brfss_key_cols)} checked variables; {brfss_key_incomplete:,} records "
    f"have a blank or nonresponse code in at least one of them.",
    f"Duplicates: {brfss_duplicates['exact_duplicates']} exact duplicate rows, "
    f"{brfss_duplicates['id_duplicates']} repeated (_STATE, SEQNO) pairs.",
    "Limitation: nonresponse codes were checked for the variables listed above, not all 301 columns, "
    "and blanks cannot be split into 'not asked' and 'missing'.",
])

Summary - CDC BRFSS 2024
 - Missing: 66,613,951 cells (48.36%), in 249 of 301 columns (2 columns entirely blank); blanks are documented as 'Not asked or Missing'.
 - Incomplete: 85,446 partially completed interviews; 153,750 'Don't know'/'Refused' codes in 13 checked variables; 129,571 records have a blank or nonresponse code in at least one of them.
 - Duplicates: 0 exact duplicate rows, 0 repeated (_STATE, SEQNO) pairs.
 - Limitation: nonresponse codes were checked for the variables listed above, not all 301 columns, and blanks cannot be split into 'not asked' and 'missing'.


In [52]:
# Free about 1.2 GB of memory before the next datasets.
# Re-run the loading cells of this section to get the `brfss` DataFrame back.
del brfss, is_blank, is_coded, brfss_row_missing_pct
gc.collect();

---
# Dataset 4 — EPA Fuel Economy Vehicle Data

- **Publisher:** U.S. Environmental Protection Agency (fueleconomy.gov, with the U.S. Department of Energy)
- **Official documentation:** <https://www.fueleconomy.gov/feg/ws/index.shtml>
- **Data file:** `vehicles.csv.zip`, the documented download under *"Download the Vehicle Data"* on
  that page
- **Unit of observation:** one row per vehicle record (`id`), i.e. one model-year configuration
  of make, model, engine, transmission and fuel

The documented bulk download is used instead of the REST API because the API returns one vehicle
per request. This file is updated by the publisher over time, so row counts depend on the download
date (delete the cached file in `data/` to fetch the current version).

## Step 1: Load the dataset

In [53]:
EPA_NAME = "EPA Fuel Economy Vehicle Data"
EPA_SOURCE = "https://www.fueleconomy.gov/feg/ws/index.shtml"
EPA_URL = "https://www.fueleconomy.gov/feg/epadata/vehicles.csv.zip"

epa_path = download_file(
    EPA_URL, DATA_DIR / "vehicles.csv.zip",
    manual_hint=f"open {EPA_SOURCE}, go to 'Download the Vehicle Data', download "
                "'Vehicle - CSV (Zipped)' and save it as data/vehicles.csv.zip",
)

# Only truly empty fields are read as missing; text such as "NA" or "None" is kept as written
epa = pd.read_csv(epa_path, low_memory=False, keep_default_na=False, na_values=[""])

show_dimensions(epa, EPA_NAME, EPA_SOURCE)

Using cached file: data/vehicles.csv.zip (2.2 MB)
Dataset : EPA Fuel Economy Vehicle Data
Source  : https://www.fueleconomy.gov/feg/ws/index.shtml
Rows    : 50,407
Columns : 84


In [54]:
show_columns(epa)

Data types: {'float64': 30, 'int64': 27, 'object': 26, 'bool': 1}

All 84 column names:
barrels08 (float64), barrelsA08 (float64), charge120 (float64), charge240 (float64), city08 (int64), city08U (float64), cityA08 (int64), cityA08U (float64), cityCD (float64), cityE (float64), cityUF (float64), co2 (int64), co2A (int64), co2TailpipeAGpm (float64), co2TailpipeGpm (float64), comb08 (int64), comb08U (float64), combA08 (int64), combA08U (float64), combE (float64), combinedCD (float64), combinedUF (float64), cylinders (object), displ (object), drive (object), engId (int64), eng_dscr (object), feScore (int64), fuelCost08 (int64), fuelCostA08 (int64), fuelType (object), fuelType1 (object), ghgScore (int64), ghgScoreA (int64), highway08 (int64), highway08U (float64), highwayA08 (int64), highwayA08U (float64), highwayCD (float64), highwayE (float64), highwayUF (float64), hlv (int64), hpv (int64), id (int64), lv2 (int64), lv4 (int64), make (object), model (object), mpgData (object), phevBlende

The CSV is read strictly: only empty fields become missing values. By default pandas would also
convert text such as `NA` into a missing value, which would hide what is actually written in the
file. One consequence is visible above: `cylinders` and `displ` are text (`object`) columns,
because a few records contain the literal text `NA` (see Step 4).

## Step 2: Representative excerpt

The dataset has 84 columns. The excerpt shows a subset that identifies each vehicle and gives its
main fuel-economy figures.

In [55]:
epa_excerpt_cols = ["id", "year", "make", "model", "VClass", "cylinders", "displ", "drive",
                    "trany", "fuelType1", "city08", "highway08", "comb08", "co2TailpipeGpm"]
show_excerpt(epa, epa_excerpt_cols)

,id,year,make,model,VClass,cylinders,displ,drive,trany,fuelType1,city08,highway08,comb08,co2TailpipeGpm
8394,17554,2002,Honda,Civic,Compact Cars,4,1.7,Front-Wheel Drive,Manual 5-spd,Regular Gasoline,28,36,31,286.677419
15711,24295,2008,Mazda,3,Compact Cars,4,2.3,Front-Wheel Drive,Manual 5-spd,Regular Gasoline,22,29,25,355.480000
22848,30813,2011,Nissan,Rogue AWD,Sport Utility Vehicle - 4WD,4,2.5,All-Wheel Drive,Automatic (variable gear ratios),Regular Gasoline,22,26,24,370.291667
33049,40074,2019,Mazda,CX-3 4WD,Compact Cars,4,2.0,4-Wheel Drive,Automatic (S6),Regular Gasoline,27,32,29,303.000000
33325,40322,2019,Ford,Mustang Convertible,Subcompact Cars,4,2.3,Rear-Wheel Drive,Manual 6-spd,Regular Gasoline,20,28,23,385.000000
33907,40849,2019,Kia,K900,Large Cars,6,3.3,All-Wheel Drive,Automatic (S8),Premium Gasoline,18,25,21,431.000000
42172,48356,2025,Genesis,GV60 AWD (19 inch Wheels),Small Sport Utility Vehicle 4WD,NaN,NaN,All-Wheel Drive,Automatic (A1),Electricity,110,90,100,0.000000
44761,50702,2027,Volvo,EX90 Twin Motor Performance (21 Inch Wheels),Standard Sport Utility Vehicle 4WD,NaN,NaN,All-Wheel Drive,Automatic (A1),Electricity,94,83,89,0.000000


## Step 3: Missing data

In [56]:
epa_missing = missing_overview(epa)

Total cells                    : 4,234,188
Total missing values           : 543,853 (12.84% of all cells)
Columns with missing values    : 18 of 84
Columns that are entirely empty: 0


In [57]:
# All columns that contain at least one missing value
epa_missing_table = missing_table(epa)
epa_missing_table[epa_missing_table["missing_count"] > 0]

,missing_count,missing_pct
c240bDscr,50272,99.73
c240Dscr,50266,99.72
sCharger,49240,97.68
rangeA,48397,96.01
fuelType2,48394,96.01
guzzler,47548,94.33
evMotor,46771,92.79
atvType,43547,86.39
tCharger,38620,76.62
trans_dscr,35363,70.15


Several of these columns are described in the documentation as flags or as fields that apply only
to some vehicles, so an empty field is expected there:

- `guzzler` ("if G or T, this vehicle is subject to the gas guzzler tax"), `tCharger` ("if T, this
  vehicle is turbocharged"), `sCharger` ("if S, …supercharged");
- `fuelType2`, `rangeA`, `atvType`, `evMotor`, `c240Dscr`, `c240bDscr` apply to dual-fuel,
  alternative-fuel or electric vehicles;
- `startStop` is documented as "Y, N, or blank for older vehicles".

The cells below check two of these against the data: which fuel types have no `cylinders` value,
and which model years have a blank `startStop`.

In [58]:
# Fuel type of the vehicles that have no cylinders value
epa.loc[epa["cylinders"].isna(), "fuelType1"].value_counts(dropna=False).to_frame("records without cylinders")

,records without cylinders
Electricity,1608
Hydrogen,42
NaN,2


In [59]:
blank_startstop_years = epa.loc[epa["startStop"].isna(), "year"]
print(f"startStop is blank for {len(blank_startstop_years):,} records, "
      f"model years {blank_startstop_years.min()} to {blank_startstop_years.max()}")
print(f"Model years in the whole file: {epa['year'].min()} to {epa['year'].max()}")

startStop is blank for 31,689 records, model years 1984 to 2012
Model years in the whole file: 1984 to 2027


## Step 4: Incomplete data

The documentation defines **`-1` = "Not available"** for the EPA scores (`feScore`, `ghgScore`,
`ghgScoreA`) and for tailpipe CO₂ from EPA tests (`co2`, `co2A`; footnote 5: tested values exist
for model year 2013 and later). These are numbers, so pandas does not count them as missing.

In [60]:
NA_CODE = {-1: "Not available"}
EPA_CODES = {"feScore": NA_CODE, "ghgScore": NA_CODE, "ghgScoreA": NA_CODE, "co2": NA_CODE, "co2A": NA_CODE}
epa_code_table = special_code_table(epa, EPA_CODES)
epa_code_table

,column,code,documented meaning,count,pct_of_rows
0,feScore,-1,Not available,32012,63.51
1,ghgScore,-1,Not available,32012,63.51
2,ghgScoreA,-1,Not available,49624,98.45
3,co2,-1,Not available,31939,63.36
4,co2A,-1,Not available,49700,98.60


In [61]:
# Model years of the records where the tested CO2 value is "not available"
co2_na_years = epa.loc[epa["co2"] == -1, "year"]
print(f"co2 == -1 for model years {co2_na_years.min()} to {co2_na_years.max()}")

co2 == -1 for model years 1984 to 2012


The documentation also states (footnote 3) that *unrounded* MPG values "are not available for some
vehicles", without naming a placeholder. In the file these fields contain `0` for many records
while the rounded MPG is never 0. The zeros are reported here as observed; the documentation does
not explicitly define `0` as a code.

In [62]:
for col in ["city08", "city08U", "highway08", "highway08U", "comb08", "comb08U"]:
    n_zero = int((epa[col] == 0).sum())
    print(f"{col:<11} == 0 : {n_zero:>6,} records ({n_zero / len(epa) * 100:.1f}%)")

city08      == 0 :      0 records (0.0%)
city08U     == 0 : 29,656 records (58.8%)
highway08   == 0 :      0 records (0.0%)
highway08U  == 0 : 29,656 records (58.8%)
comb08      == 0 :      0 records (0.0%)
comb08U     == 0 : 29,656 records (58.8%)


Finally, the text columns are checked for written-out placeholders. These are neither blank cells
nor documented codes; they are reported as found.

In [63]:
epa_placeholders = placeholder_table(epa)
epa_placeholders

,column,text value,count
0,cylinders,NA,3
1,displ,NA,2
2,eng_dscr,NONE,1


In [64]:
# The records behind the placeholder text in cylinders / displ
epa.loc[epa["cylinders"].isin(["NA"]) | epa["displ"].isin(["NA"]),
        ["id", "year", "make", "model", "fuelType1", "cylinders", "displ"]]

,id,year,make,model,fuelType1,cylinders,displ
21410,29472,1985,Subaru,RX Turbo,Regular Gasoline,NA,NA
21411,29473,1985,Subaru,RX Turbo,Regular Gasoline,NA,NA
21500,29557,1986,Mazda,RX-7,Regular Gasoline,NA,1.3


In [65]:
# Records lacking a basic descriptive field (actual missing values)
epa_key_cols = ["year", "make", "model", "VClass", "drive", "trany", "fuelType1", "comb08"]
epa_key_missing = epa[epa_key_cols].isna().sum()
epa_key_incomplete = int(epa[epa_key_cols].isna().any(axis=1).sum())
epa_co2_na = int((epa["co2"] == -1).sum())
epa_unrounded_zero = int((epa["comb08U"] == 0).sum())

print(epa_key_missing.to_string())
print(f"\nRecords missing at least one of these fields: {epa_key_incomplete:,}")

year            0
make            0
model           0
VClass          0
drive        1186
trany          11
fuelType1       2
comb08          0

Records missing at least one of these fields: 1,196


**Legitimate values that may look unusual.** Very high MPG values belong to electric vehicles: the
documentation (footnote 11) states that for electric and CNG vehicles the figure is MPGe. Model
years later than the current calendar year are upcoming models already listed by the publisher.

In [66]:
print("Highest combined MPG by fuel type (top 3):")
print(epa.groupby("fuelType1")["comb08"].max().sort_values(ascending=False).head(3).to_string())

Highest combined MPG by fuel type (top 3):
fuelType1
Electricity         146
Hydrogen             74
Regular Gasoline     59


## Step 5: Duplicate data

`id` is documented as the vehicle record id. Because every row has its own `id`, an *exact*
duplicate row can only occur if an id is repeated. A second check therefore ignores `id` and looks
for records that are otherwise identical.

In [67]:
epa_duplicates = duplicate_overview(epa, id_columns=["id"])

epa_same_except_id = int(epa.drop(columns=["id"]).duplicated().sum())
print(f"Rows identical to an earlier row in every column except id: {epa_same_except_id:,}")

Exact duplicate rows: 0 (0.0% of rows)
Rows repeating an earlier identifier ['id']: 0
Rows identical to an earlier row in every column except id: 40


In [68]:
# A few of the records that are identical except for their id
same_except_id = epa[epa.drop(columns=["id"]).duplicated(keep=False)]
same_except_id.sort_values(["year", "make", "model", "id"])[epa_excerpt_cols].head(6)

,id,year,make,model,VClass,cylinders,displ,drive,trany,fuelType1,city08,highway08,comb08,co2TailpipeGpm
19211,27459,1984,Jeep,J-10 Pickup 4WD,Standard Pickup Trucks 4WD,8,5.9,4-Wheel or All-Wheel Drive,Automatic 3-spd,Regular Gasoline,10,12,11,807.909091
19213,27460,1984,Jeep,J-10 Pickup 4WD,Standard Pickup Trucks 4WD,8,5.9,4-Wheel or All-Wheel Drive,Automatic 3-spd,Regular Gasoline,10,12,11,807.909091
7511,1676,1985,Subaru,Sedan 4WD,Special Purpose Vehicles,4,1.8,4-Wheel or All-Wheel Drive,Automatic 3-spd,Regular Gasoline,21,23,22,403.954545
7522,1677,1985,Subaru,Sedan 4WD,Special Purpose Vehicles,4,1.8,4-Wheel or All-Wheel Drive,Automatic 3-spd,Regular Gasoline,21,23,22,403.954545
20879,2897,1987,CX Automotive,CX 25Tri,Two Seaters,4,2.5,Rear-Wheel Drive,Manual 5-spd,Regular Gasoline,15,19,17,522.764706
20890,2898,1987,CX Automotive,CX 25Tri,Two Seaters,4,2.5,Rear-Wheel Drive,Manual 5-spd,Regular Gasoline,15,19,17,522.764706


Records that match in every published column but have different ids are not necessarily errors:
the file does not contain every attribute that can distinguish two vehicle configurations, so
these are reported as *potential* duplicates only. Many rows also share make, model and year by
design, because each engine/transmission/fuel combination has its own record.

## Step 6: Summary of findings

In [69]:
record_results(EPA_NAME, epa, epa_missing, epa_duplicates, [
    f"Missing: {epa_missing['total_missing']:,} cells ({epa_missing['missing_pct']}%), in "
    f"{epa_missing['columns_with_missing']} of {epa.shape[1]} columns, mostly flag or "
    f"vehicle-type-specific fields that are blank when not applicable.",
    f"Incomplete: documented -1 = 'Not available' in the score and tested-CO2 fields "
    f"(co2 = -1 for {epa_co2_na:,} records); unrounded combined MPG is 0 for {epa_unrounded_zero:,} records; "
    f"{int(epa_placeholders['count'].sum())} cells contain placeholder text such as 'NA'; "
    f"{epa_key_incomplete:,} records lack a basic descriptive field.",
    f"Duplicates: {epa_duplicates['exact_duplicates']} exact duplicate rows, "
    f"{epa_duplicates['id_duplicates']} repeated ids; {epa_same_except_id} rows identical to another except for id.",
    "Limitation: the file is updated by the publisher, so counts depend on the download date; "
    "0 is not explicitly documented as a 'not available' code.",
])

Summary - EPA Fuel Economy Vehicle Data
 - Missing: 543,853 cells (12.84%), in 18 of 84 columns, mostly flag or vehicle-type-specific fields that are blank when not applicable.
 - Incomplete: documented -1 = 'Not available' in the score and tested-CO2 fields (co2 = -1 for 31,939 records); unrounded combined MPG is 0 for 29,656 records; 6 cells contain placeholder text such as 'NA'; 1,196 records lack a basic descriptive field.
 - Duplicates: 0 exact duplicate rows, 0 repeated ids; 40 rows identical to another except for id.
 - Limitation: the file is updated by the publisher, so counts depend on the download date; 0 is not explicitly documented as a 'not available' code.


---
# Dataset 5 — Amazon Purchase Histories (Open e-commerce 1.0)

- **Publisher:** Harvard Dataverse (original research repository)
- **Official source:** <https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY>
- **Data file:** `amazon-purchases.csv`, retrieved through the Dataverse native API
- **Unit of observation:** one row per purchased item line in a participant's Amazon order history;
  each participant (`Survey ResponseID`) has many rows by design

The repository also contains `survey.csv` (participant demographics) and `fields.csv` (which
describes the survey columns). Only the purchase file is investigated here.

## Step 1: Load the dataset

The Dataverse API is first asked for the dataset's file list, which provides the file id and the
publisher's MD5 checksum. The downloaded file is checked against that checksum. No API token is
needed because the dataset is public.

In [70]:
AMZ_NAME = "Amazon Purchase Histories (Open e-commerce 1.0)"
AMZ_SOURCE = "https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY"
AMZ_API = "https://dataverse.harvard.edu/api/datasets/:persistentId/?persistentId=doi:10.7910/DVN/YGLYDY"
AMZ_FILE = "amazon-purchases.csv"
AMZ_HINT = f"open {AMZ_SOURCE}, download {AMZ_FILE} and save it as data/{AMZ_FILE}"

# Look up the file in the dataset's official file list.
# The request names this notebook as the client; Dataverse rejects the default "python-requests" name.
AMZ_HEADERS = {"User-Agent": "dataset-quality-investigation-notebook (python-requests)"}
try:
    response = requests.get(AMZ_API, headers=AMZ_HEADERS, timeout=60)
    response.raise_for_status()
    files = response.json()["data"]["latestVersion"]["files"]
except Exception as err:
    raise RuntimeError(f"Could not read the Dataverse file list: {err}\nManual download: {AMZ_HINT}") from err

amz_file_info = next(f["dataFile"] for f in files if f["dataFile"]["filename"] == AMZ_FILE)
print(f"File id: {amz_file_info['id']} | size: {amz_file_info['filesize'] / 1e6:,.1f} MB | md5: {amz_file_info['md5']}")

File id: 7616235 | size: 313.1 MB | md5: 3549d72735b1211ab009ff0177b94bd9


In [71]:
amz_path = download_file(
    f"https://dataverse.harvard.edu/api/access/datafile/{amz_file_info['id']}",
    DATA_DIR / AMZ_FILE, manual_hint=AMZ_HINT,
)

# Compare the local file with the checksum published by Dataverse
md5 = hashlib.md5()
with open(amz_path, "rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
        md5.update(block)
assert md5.hexdigest() == amz_file_info["md5"], "Checksum mismatch: delete the file in data/ and download it again"
print("Checksum matches the published MD5.")

Using cached file: data/amazon-purchases.csv (313.1 MB)
Checksum matches the published MD5.


In [72]:
# Read strictly, as for the EPA file: only empty fields become missing values.
# Product codes are read as text so that they are not converted to numbers.
amz = pd.read_csv(amz_path, keep_default_na=False, na_values=[""],
                  dtype={"ASIN/ISBN (Product Code)": str})

show_dimensions(amz, AMZ_NAME, AMZ_SOURCE)
print(f"Memory  : {amz.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

Dataset : Amazon Purchase Histories (Open e-commerce 1.0)
Source  : https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YGLYDY
Rows    : 1,850,717
Columns : 8
Memory  : 951 MB


In [73]:
show_columns(amz)

Data types: {'object': 6, 'float64': 2}


,dtype
Order Date,object
Purchase Price Per Unit,float64
Quantity,float64
Shipping Address State,object
Title,object
ASIN/ISBN (Product Code),object
Category,object
Survey ResponseID,object


## Step 2: Representative excerpt

The file has only 8 columns, so all of them are shown. Long product titles are shortened *on
screen* with "..." to keep the table readable; the stored values are unchanged.

In [74]:
show_excerpt(amz, list(amz.columns))

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
147918,2019-05-14,22.99,1.0,VA,NaN,B07DXGBKRH,NaN,R_1KkFs7rgWXlWf58
425490,2022-06-07,3.99,1.0,MI,"Garnier Fructis Sleek & Shine Anti-Frizz Serum, Frizzy, ...",B006J44BSI,HAIR_STYLING_AGENT,R_20Z3XrmdVrFDp3r
651418,2020-08-03,19.99,1.0,TX,"Elmer's Clear Liquid School Glue, Slime Glue, & Craft Gl...",B071JPD9M3,BONDING_ADHESIVES,R_2SqWTZJSEFu0jiK
816975,2018-10-09,15.99,1.0,MN,LEGO Technic WHACK! 42072 Building Kit with Pull Back To...,B075QYRTXZ,TOY_BUILDING_BLOCK,R_2pKbiSIpze2jP6t
881377,2020-01-12,13.98,1.0,NY,"Huggies Natural Care Sensitive Baby Wipes, Unscented, 48...",B07M6XVCBY,SKIN_CLEANING_WIPE,R_2uyQHwENySBamVs
1136729,2021-11-16,6.52,1.0,FL,"Einstein Bros. Bagels, Take & Toast, Asiago, 18.5 oz (5 ct)",B08QTQZYFV,BREAD,R_3NxQLbGnG7D6JNk
1265335,2018-01-27,7.98,1.0,PA,NaN,B008ZZC6S6,NaN,R_3k5JabbpW5kTO7M
1473301,2021-02-06,9.99,1.0,CT,ZEBRE Case Compatible with Apple Watch Screen Protector ...,B08Q7Q2KRC,SCREEN_PROTECTOR,R_AhNiHUaRBzQHY1b


## Step 3: Missing data

In [75]:
amz_missing = missing_overview(amz)

Total cells                    : 14,805,736
Total missing values           : 267,983 (1.81% of all cells)
Columns with missing values    : 4 of 8
Columns that are entirely empty: 0


In [76]:
amz_missing_table = missing_table(amz)
amz_missing_table

,missing_count,missing_pct
Title,89740,4.85
Category,89458,4.83
Shipping Address State,87812,4.74
ASIN/ISBN (Product Code),973,0.05
Order Date,0,0.00
Purchase Price Per Unit,0,0.00
Quantity,0,0.00
Survey ResponseID,0,0.00


In [77]:
# Do the missing values occur together in the same records?
title_missing = amz["Title"].isna()
category_missing = amz["Category"].isna()
state_missing = amz["Shipping Address State"].isna()

print(f"Title and Category both missing : {int((title_missing & category_missing).sum()):,}")
print(f"Title missing, Category present : {int((title_missing & ~category_missing).sum()):,}")
print(f"Category missing, Title present : {int((category_missing & ~title_missing).sum()):,}")

print("\nNumber of missing fields per record:")
print(amz.isna().sum(axis=1).value_counts().sort_index().to_string())

Title and Category both missing : 88,832
Title missing, Category present : 908
Category missing, Title present : 626

Number of missing fields per record:
0    1675015
1      86166
2      86834
3       2659
4         43


In [78]:
# The product categories that most often have no shipping state
amz.loc[state_missing, "Category"].value_counts(dropna=False).head(5).to_frame("records without shipping state")

,records without shipping state
GIFT_CARD,24719
DOWNLOADABLE_VIDEO_GAME,5627
ABIS_GIFT_CARD,5402
ELECTRONIC_GIFT_CARD,2975
ABIS_BOOK,2728


The repository description lists the columns of this file but does not document why a field may be
empty, so the reasons for these blanks cannot be confirmed from the official documentation. The
figures above only describe where the blanks occur.

## Step 4: Incomplete data

Unlike the survey datasets, this file has **no documented special codes** for unknown or refused
values. The checks below therefore look for (a) text placeholders that could stand in for a missing
value, (b) non-positive prices or quantities, and (c) records that lack the fields needed to
identify the product.

In [79]:
# (a) Text placeholders in the text columns
amz_placeholders = placeholder_table(amz)
print("Placeholder-like text values found:", int(amz_placeholders["count"].sum()))
amz_placeholders

Placeholder-like text values found: 1


,column,text value,count
0,Title,?,1


A match here is not automatically an unknown value: a short text such as `?` can also be the real
title of a product. Any match is listed so that it can be judged individually.

In [80]:
# (b) Prices and quantities
print("Purchase Price Per Unit <= 0 :", f"{int((amz['Purchase Price Per Unit'] <= 0).sum()):,}")
print("Quantity <= 0                :", f"{int((amz['Quantity'] <= 0).sum()):,}")
amz[["Purchase Price Per Unit", "Quantity"]].agg(["min", "max"])

Purchase Price Per Unit <= 0 : 0
Quantity <= 0                : 0


,Purchase Price Per Unit,Quantity
min,0.01,1.0
max,6398.95,339.0


In [81]:
# (c) Records that cannot be fully described
amz_no_product_info = int((title_missing & category_missing).sum())
amz_no_product_code = int(amz["ASIN/ISBN (Product Code)"].isna().sum())
amz_any_missing = int(amz.isna().any(axis=1).sum())

print(f"Records with neither Title nor Category     : {amz_no_product_info:,}")
print(f"Records without a product code              : {amz_no_product_code:,}")
print(f"Records with at least one missing field     : {amz_any_missing:,} "
      f"({amz_any_missing / len(amz) * 100:.2f}%)")

Records with neither Title nor Category     : 88,832
Records without a product code              : 973
Records with at least one missing field     : 175,702 (9.49%)


**Legitimate values that may look unusual.** Two further observations are reported without
judgement, because the documentation does not settle how to interpret them:

- The repository describes the data as "spanning 2018 through 2022". The order dates in the file
  are counted by year below.
- Amazon ASINs and ISBN-10s are 10 characters long. Product codes of other lengths are counted.

In [82]:
order_year = amz["Order Date"].str[:4]
print("Order Date range:", amz["Order Date"].min(), "to", amz["Order Date"].max())
print(order_year.value_counts().sort_index().to_string())

amz_after_2022 = int((order_year > "2022").sum())
print(f"\nRecords dated after 2022: {amz_after_2022:,} ({amz_after_2022 / len(amz) * 100:.2f}%)")

Order Date range: 2018-01-01 to 2024-08-15
2018    225465
2019    266016
2020    409212
2021    467870
2022    442230
2023     39923
2024         1

Records dated after 2022: 39,924 (2.16%)


In [83]:
code_length = amz["ASIN/ISBN (Product Code)"].str.len()
amz_short_codes = int((code_length < 10).sum())
print(code_length.value_counts(dropna=False).sort_index().rename("records").rename_axis("code length").to_string())

code length
7.0           7
8.0         377
9.0        3007
10.0    1846353
NaN         973


## Step 5: Duplicate data

This file has **no order id or line id**. `Survey ResponseID` identifies the participant, not the
purchase, so it repeats on every row belonging to that participant by design and is not a
duplicate indicator.

In [84]:
amz_duplicates = duplicate_overview(amz)

orders_per_participant = amz["Survey ResponseID"].value_counts()
print(f"Participants (unique Survey ResponseID): {len(orders_per_participant):,}")
print(f"Rows per participant: median {orders_per_participant.median():.0f}, "
      f"min {orders_per_participant.min()}, max {orders_per_participant.max():,}")

Exact duplicate rows: 11,624 (0.628% of rows)
Participants (unique Survey ResponseID): 5,027
Rows per participant: median 232, min 1, max 5,415


In [85]:
# What kinds of purchases are repeated exactly? (all rows that belong to a duplicate group)
amz_dup_rows = amz[amz.duplicated(keep=False)]
amz_dup_rows["Category"].value_counts(dropna=False).head(5).to_frame("rows in exact-duplicate groups")

,rows in exact-duplicate groups
GIFT_CARD,6258
ABIS_GIFT_CARD,1433
ABIS_BOOK,838
PET_FOOD,553
ELECTRONIC_GIFT_CARD,530


In [86]:
# One example group of identical rows
example = amz_dup_rows.sort_values(list(amz.columns)).head(4)
example

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
1176133,2018-01-01,0.15,2.0,NJ,NaN,B07758RF83,NaN,R_3R3FfvTaRKxigNl
1176142,2018-01-01,0.15,2.0,NJ,NaN,B07758RF83,NaN,R_3R3FfvTaRKxigNl
682365,2018-01-01,0.50,1.0,NaN,Amazon.com Gift Card Balance Reload,B00IX1I3G6,GIFT_CARD,R_2VwjXhJ5OznStEi
682366,2018-01-01,0.50,1.0,NaN,Amazon.com Gift Card Balance Reload,B00IX1I3G6,GIFT_CARD,R_2VwjXhJ5OznStEi


An exact duplicate row here means the same participant bought the same product at the same price
and quantity, shipped to the same state, on the same day. Without an order id this cannot be
separated into (a) a genuine repeat purchase or two separate order lines and (b) a record that was
exported twice. The duplicate rows are therefore *potential* duplicates and are left in place.

## Step 6: Summary of findings

In [87]:
record_results(AMZ_NAME, amz, amz_missing, amz_duplicates, [
    f"Missing: {amz_missing['total_missing']:,} cells ({amz_missing['missing_pct']}%), in "
    f"{amz_missing['columns_with_missing']} of {amz.shape[1]} columns (Title, Category, shipping state, product code); "
    f"{amz_any_missing:,} records have at least one missing field.",
    f"Incomplete: no documented special codes; placeholder-like text values found: {int(amz_placeholders['count'].sum())}; "
    f"{amz_no_product_info:,} records have neither Title nor Category; {amz_after_2022:,} records are dated after 2022; "
    f"{amz_short_codes:,} product codes are shorter than 10 characters.",
    f"Duplicates: {amz_duplicates['exact_duplicates']:,} exact duplicate rows "
    f"({amz_duplicates['exact_duplicates_pct']}%); with no order id they cannot be told apart from repeat purchases.",
    "Limitation: the reasons for blank fields are not documented; participant survey data (survey.csv) was not examined.",
])

Summary - Amazon Purchase Histories (Open e-commerce 1.0)
 - Missing: 267,983 cells (1.81%), in 4 of 8 columns (Title, Category, shipping state, product code); 175,702 records have at least one missing field.
 - Incomplete: no documented special codes; placeholder-like text values found: 1; 88,832 records have neither Title nor Category; 39,924 records are dated after 2022; 3,391 product codes are shorter than 10 characters.
 - Duplicates: 11,624 exact duplicate rows (0.628%); with no order id they cannot be told apart from repeat purchases.
 - Limitation: the reasons for blank fields are not documented; participant survey data (survey.csv) was not examined.


---
# Final comparison

All figures in this table were calculated above from the complete downloaded files.

In [88]:
comparison = pd.DataFrame(RESULTS)
assert len(comparison) == 5, "Run all five dataset sections before building the comparison table"

with pd.option_context("display.max_colwidth", None):
    display(comparison[["Dataset", "Rows", "Columns", "Missing Values (%)", "Exact Duplicate Rows"]])

,Dataset,Rows,Columns,Missing Values (%),Exact Duplicate Rows
0,"NHANES Body Measures (BMX_L, 2021-2023)",8860,22,54.91,0
1,"NHANES Dietary Interview - Total Nutrient Intakes, First Day (DR1TOT_L, 2021-2023)",8860,168,49.71,0
2,CDC BRFSS 2024,457670,301,48.36,0
3,EPA Fuel Economy Vehicle Data,50407,84,12.84,0
4,Amazon Purchase Histories (Open e-commerce 1.0),1850717,8,1.81,11624


In [89]:
# The complete table, including the observations column in full
display(comparison.style
        .format({"Rows": "{:,}", "Columns": "{:,}", "Missing Values (%)": "{:.2f}", "Exact Duplicate Rows": "{:,}"})
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
        .hide(axis="index"))

Dataset,Rows,Columns,Missing Values (%),Exact Duplicate Rows,Main Data-Quality Observations
"NHANES Body Measures (BMX_L, 2021-2023)","8,860",22,54.91,0,"Missing: 107,023 cells (54.91%), in 20 of 22 columns; most are in comment columns and in measurements with a documented age-restricted target. Incomplete: 625 records have an exam status other than complete; 1,603 'Could not obtain' codes; 389 records lack weight, height or BMI. Duplicates: 0 exact duplicate rows, 0 repeated SEQN. Limitation: the file has no age variable, so age-ineligible blanks cannot be separated from unobtained measurements without the NHANES demographics file."
"NHANES Dietary Interview - Total Nutrient Intakes, First Day (DR1TOT_L, 2021-2023)","8,860",168,49.71,0,"Missing: 739,930 cells (49.71%), in 165 of 168 columns; nutrient totals are blank for 2,166 records, and many follow-up questions are blank through documented skips. Incomplete: 2,166 records have a recall status other than reliable (2,106 have day-one weight 0 = not done/incomplete); 627 'Refused'/'Don't know' codes in the checked columns. Duplicates: 0 exact duplicate rows, 0 repeated SEQN. Limitation: special codes were checked for the questionnaire columns listed above, not for all 168 columns."
CDC BRFSS 2024,"457,670",301,48.36,0,"Missing: 66,613,951 cells (48.36%), in 249 of 301 columns (2 columns entirely blank); blanks are documented as 'Not asked or Missing'. Incomplete: 85,446 partially completed interviews; 153,750 'Don't know'/'Refused' codes in 13 checked variables; 129,571 records have a blank or nonresponse code in at least one of them. Duplicates: 0 exact duplicate rows, 0 repeated (_STATE, SEQNO) pairs. Limitation: nonresponse codes were checked for the variables listed above, not all 301 columns, and blanks cannot be split into 'not asked' and 'missing'."
EPA Fuel Economy Vehicle Data,"50,407",84,12.84,0,"Missing: 543,853 cells (12.84%), in 18 of 84 columns, mostly flag or vehicle-type-specific fields that are blank when not applicable. Incomplete: documented -1 = 'Not available' in the score and tested-CO2 fields (co2 = -1 for 31,939 records); unrounded combined MPG is 0 for 29,656 records; 6 cells contain placeholder text such as 'NA'; 1,196 records lack a basic descriptive field. Duplicates: 0 exact duplicate rows, 0 repeated ids; 40 rows identical to another except for id. Limitation: the file is updated by the publisher, so counts depend on the download date; 0 is not explicitly documented as a 'not available' code."
Amazon Purchase Histories (Open e-commerce 1.0),"1,850,717",8,1.81,"11,624","Missing: 267,983 cells (1.81%), in 4 of 8 columns (Title, Category, shipping state, product code); 175,702 records have at least one missing field. Incomplete: no documented special codes; placeholder-like text values found: 1; 88,832 records have neither Title nor Category; 39,924 records are dated after 2022; 3,391 product codes are shorter than 10 characters. Duplicates: 11,624 exact duplicate rows (0.628%); with no order id they cannot be told apart from repeat purchases. Limitation: the reasons for blank fields are not documented; participant survey data (survey.csv) was not examined."


In [90]:
# Save the comparison table next to the notebook for use in the paper
comparison.to_csv("dataset_quality_comparison.csv", index=False)
print("Saved dataset_quality_comparison.csv")

Saved dataset_quality_comparison.csv


## How to read the comparison

The two numeric quality columns do not mean the same thing in every dataset, so they should not be
ranked against each other directly.

**Missing Values (%)** is the share of empty cells in the whole table.

- *NHANES and BRFSS:* most empty cells are structural. Questions and measurements apply only to
  certain ages or follow from earlier answers, optional BRFSS modules are not used by every state,
  and NHANES comment columns are filled only when there is a comment. A high percentage mainly
  reflects the survey design and the number of conditional columns.
- *EPA:* most empty cells are flag fields or fields that apply only to some vehicle types.
- *Amazon:* every column applies to every row, so an empty cell is an absent piece of information,
  although the cause is not documented.

**The percentage does not include coded nonresponse.** In the survey datasets, "Refused" and
"Don't know" are stored as numbers (7/9, 77/99, 7777/9999), and the EPA file uses -1 for "Not
available" (plus the text `NA` in a few cells). These are counted in Step 4 of each section, not in the missing-value percentage.

**Exact Duplicate Rows** also depends on the structure of the file.

- *NHANES, BRFSS and EPA* have a record identifier (`SEQN`, `_STATE` + `SEQNO`, `id`), so an exact
  duplicate would indicate a repeated record.
- *Amazon* has no order identifier and many rows per participant by design, so identical rows may
  be genuine repeat purchases.

**Scope of these results.** Special codes were checked for the documented variables listed in each
section, not for every column of the wide survey files. No data were cleaned, imputed or removed.